## **<u>Context Manager</u>**
it's a way to ensure that certain code runs before a block of code executes, and other cleanup code runs after, no matter what happens even if an exception is raised."
The primary need is for robust resource management and ensuring deterministic cleanup.

The Problem (Without a Context Manager):
- Verbose: You have to write the same boilerplate code every time.
- Error-prone: It's easy to forget the finally block.
- Less Readable: The core logic is nested inside the error-handling code.

The Solution (With a Custom Context Manager):
- Cleanup Guarantee: The teardown code (closing, releasing) always runs.
- Exception Safety: Your resources won't leak if your code encounters an error.
- DRY Principle: You write the setup/teardown logic once and reuse it.
- Clarity & Readability: The with block clearly defines the scope of the resource.

#### **<u>Structure</u>**
__enter__(self): Runs when the with block is entered. Its return value is bound to the as variable.

__exit__(self, exc_type, exc_val, exc_tb): Runs when the with block is exited. It handles any exceptions that occurred.

In [ ]:
# Creating custon context mnagers
class MyContextManager:
    def __enter__(self): # This must be the first function, even before constructor defination
        # Here you menation all the variables and call the methods require to run this context manager 
        print("Entering the context")
        # Variables and Methods
        return self # This bound to a `as` variable in the context manager
    
    # The main code logic
    def __init__(self, name, id):
        self.id = id
        print(name, id)
    
    @property
    def getter(self):
        return self.id
    @getter.setter
    def getter(self, val):
        return val + 10

    def __exit__(self, exc_type, exc_value, traceback): # `exc_type, exc_value, traceback` - maindatory
        # Here you do stuff like closing the db connection (teardown actions) and more 
        print("Exiting the context")
        # The exception details are passed to the arguments if an exception occurs

In [2]:
# Using the custom context manager
with MyContextManager('dex', 19) as cm:
    print("Inside the context")
    print(cm.getter) # cm is self retured at __enter__

dex 19
Entering the context
Inside the context
19
Exiting the context


#### The execution order
1. The constructor<br>
2. The __enter__ method<br>
3. The main code<br>
4. The __exit__ method

In [3]:
# Creating a normal object - It runs just like the normal class
obj = MyContextManager('sam', 21)

sam 21


In [13]:
# Let's create a context manager to time a block of code.
import time
class Timer:
    def __enter__(self):
        self.current_time = time.perf_counter()

    def __exit__(self, exc_type, exc_value, traceback):
        self.total_time = time.perf_counter() - self.current_time
        print(f"Total Time require to execute your code is {self.total_time}")

In [14]:
with Timer() as t:
    sum([1, 2, 3, 4, 5])

Total Time require to execute your code is 5.199999577598646e-06


The three parameters — `exc_type`, `exc_value`, and `traceback` are mandatory because they allow the context manager to handle errors that might occur inside the with block.

#### What do they represent?

- exc_type: The class of the exception (e.g., ZeroDivisionError).
- exc_value: The actual exception instance (e.g., the error message like "division by zero").
- traceback: A traceback object which contains the call stack information (where exactly the error happened).
How do they behave?
<br><br>
→ If NO exception occurs: All three parameters are None.<br>
→ If an exception occurs: Python populates these three variables with the error details.<br>
→ If you return True from __exit__, the exception is "swallowed" (suppressed), and your program continues.<br>
→ If you return False (or nothing), the exception is re-raised after __exit__ finishes.

In [ ]:
class ErrorHandler:
    def __enter__(self):
        print("--- Entering the block ---")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("--- Exiting the block ---")
        
        if exc_type:
            try:
                print(f"Error caught!")
                print(f"Type: {exc_type}")
                print(f"Value: {exc_value}")
                # Returning True suppresses the error
                return True 
            except:
                print("Error is not resolved")
        
        print("No errors occurred.")
        return False

In [7]:
# Scenario 1: No Error
with ErrorHandler():
    print("Action: Doing something safe.")

--- Entering the block ---
Action: Doing something safe.
--- Exiting the block ---
No errors occurred.


In [6]:
# Scenario 2: With Error
with ErrorHandler():
    print("Action: Attempting to divide by zero...")
    result = 1 / 0  # This will trigger the exception

print("\nProgram continues safely because we returned True in __exit__.")

--- Entering the block ---
Action: Attempting to divide by zero...
--- Exiting the block ---
Error caught!
Type: <class 'ZeroDivisionError'>
Value: division by zero

Program continues safely because we returned True in __exit__.
